In [ ]:
# --- BƯỚC 0: SETUP ---
!pip uninstall -y tensorflow pyarrow
!pip install "pyarrow<20.0.0"
!pip install -U -q git+https://github.com/illuin-tech/colpali
!pip install -U -q transformers accelerate peft bitsandbytes

import torch
import warnings
import os
import gc
warnings.filterwarnings('ignore')

In [ ]:
# --- BƯỚC 1: SETUP DATA ---
import glob
import os
import pandas as pd
import numpy as np
import json
from tqdm.notebook import tqdm

# chỉ số mốc layout
START_IDX = 5
END_IDX = 10

SEARCH_ROOT = "/kaggle/input"
ANNOTATIONS_PATH = "/kaggle/input/mmdocir-eval-data/MMDocIR_annotations.jsonl"
PARQUET_PATH = "/kaggle/input/mmdocir-eval-data/MMDocIR_layouts.parquet"
ENHANCED_JSONL_DIR = None
ENHANCED_IMG_DIR = None

for root, dirs, files in os.walk(SEARCH_ROOT):
    if "LAYOUT_CONTENT_FINAL" in root: ENHANCED_JSONL_DIR = root
    if "IMAGE ENHACED" in root or (sum(1 for f in files if f.endswith(".jpg")) > 1000): 
        ENHANCED_IMG_DIR = root

if not ENHANCED_JSONL_DIR: ENHANCED_JSONL_DIR = "/kaggle/input/siglip-qwen-enhaced/SIGLIP_QWEN_ENHACED/LAYOUT_CONTENT_FINAL"
if not ENHANCED_IMG_DIR: ENHANCED_IMG_DIR = "/kaggle/input/siglip-qwen-enhaced/SIGLIP_QWEN_ENHACED/IMAGE ENHACED"

print(f"    Enhanced JSONL: {ENHANCED_JSONL_DIR}")

#  SORTED
# A. Lấy danh sách doc có câu hỏi
valid_docs_in_qa = set()
with open(ANNOTATIONS_PATH, 'r') as f:
    for line in f:
        try:
            d = json.loads(line)
            valid_docs_in_qa.add(d['doc_name'].replace('.pdf', ''))
        except: pass

# B. Lấy danh sách doc có trong folder JSONL
available_jsonls = glob.glob(os.path.join(ENHANCED_JSONL_DIR, "*.jsonl"))
jsonl_map = {} 
for p in available_jsonls:
    fname = os.path.basename(p).replace('_layout.jsonl', '')
    jsonl_map[fname] = p

intersection_docs = sorted(list(valid_docs_in_qa.intersection(jsonl_map.keys())))

if not intersection_docs:
    raise ValueError("Lỗi: Không tìm thấy tài liệu chung nào!")

print(f"    -> Found {len(intersection_docs)} docs valid.")

# CẮT BATCH THEO INDEX
target_doc_names = intersection_docs[START_IDX:END_IDX]
target_files = [jsonl_map[d] for d in target_doc_names]

print(f" PROCESSING BATCH [{START_IDX}:{END_IDX}]")
print(f"    -> Selected Docs: {target_doc_names}")
if len(target_doc_names) == 0:
    print(" Batch này rỗng! Hãy kiểm tra lại index.")

# ==============================================================================
# LOADING DATAFRAME
# ==============================================================================
print("    Loading Backbone (Parquet)...")
df_orig = pd.read_parquet(PARQUET_PATH)
df_orig['join_doc_name'] = df_orig['doc_name'].str.replace('.pdf', '', regex=False)
df_orig = df_orig[df_orig['join_doc_name'].isin(target_doc_names)]

print("    Loading Enrichment (JSONL)...")
dfs = []
for f in tqdm(target_files, desc="Reading JSONLs"):
    try:
        temp = pd.read_json(f, lines=True)
        temp['join_doc_name'] = os.path.basename(f).replace('_layout.jsonl', '')
        if 'layout' in temp.columns: temp = temp.rename(columns={'layout': 'layout_id'})
        
        cols = ['join_doc_name', 'layout_id', 'vlm_text', 'img_enhanced_path']
        if 'text_level' in temp.columns: cols.append('text_level')
        
        temp = temp[ [c for c in cols if c in temp.columns] ]
        dfs.append(temp)
    except: pass

if dfs:
    df_enh = pd.concat(dfs, ignore_index=True)
    df_enh = df_enh.rename(columns={'vlm_text': 'vlm_text_enhanced', 'text_level': 'text_level_enhanced'})
else:
    df_enh = pd.DataFrame()

print("    Merging & Contextualizing...")
df_final = pd.merge(df_orig, df_enh, on=['join_doc_name', 'layout_id'], how='left')
df_final = df_final.sort_values(by=['join_doc_name', 'page_id', 'layout_id'])

# Context logic
def identify_header(row):
    if row.get('type') in ['title', 'section_header', 'header']: return str(row.get('text', ''))
    if pd.notna(row.get('text_level_enhanced')): return str(row.get('text', ''))
    return np.nan

df_final['temp_header'] = df_final.apply(identify_header, axis=1)
df_final['current_section'] = df_final.groupby('join_doc_name')['temp_header'].ffill().fillna("General Content")

# Source Selection
enh_image_map = {}
if os.path.exists(ENHANCED_IMG_DIR):
    for f in glob.glob(os.path.join(ENHANCED_IMG_DIR, "*")):
        enh_image_map[os.path.basename(f)] = f

def get_best_sources(row):
    img_type, img_data = None, None
    if pd.notna(row.get('img_enhanced_path')):
        fname = os.path.basename(str(row['img_enhanced_path']))
        if fname in enh_image_map: img_type, img_data = 'path', enh_image_map[fname]
    if img_data is None and pd.notna(row.get('image_binary')): img_type, img_data = 'binary', row['image_binary']

    raw_content = ""
    if pd.notna(row.get('vlm_text_enhanced')) and len(str(row['vlm_text_enhanced'])) > 5: raw_content = str(row['vlm_text_enhanced'])
    elif pd.notna(row.get('text')) and len(str(row['text'])) > 5: raw_content = str(row['text'])
    elif pd.notna(row.get('ocr_text')) and len(str(row['ocr_text'])) > 5: raw_content = str(row['ocr_text'])
    elif pd.notna(row.get('vlm_text')): raw_content = str(row['vlm_text'])
    
    section = row.get('current_section', '')
    final_text_prompt = f"Section: {section} \n Content: {raw_content}"
    if len(final_text_prompt) < 10: final_text_prompt = "Document layout."

    return pd.Series([img_type, img_data, final_text_prompt], index=['img_type', 'img_data', 'final_text'])

processed = df_final.apply(get_best_sources, axis=1)
sample_layouts_df = pd.concat([df_final, processed], axis=1).dropna(subset=['img_type']).reset_index(drop=True)

print("="*60)
print(f" BATCH {START_IDX}-{END_IDX} READY!")
print(f" Docs: {target_doc_names}")
print(f" Total Layouts: {len(sample_layouts_df)}")
print("="*60)

In [ ]:
# --- BƯỚC 2: LOAD MODEL ---
print(">>> BƯỚC 2: Loading ColSmolVLM-500M...")

import torch
import gc
import os
from colpali_engine.models import ColIdefics3, ColIdefics3Processor

gc.collect()
torch.cuda.empty_cache()

device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "vidore/colSmol-500M"

model = ColIdefics3.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map=device,
    attn_implementation="eager" 
).eval()

processor = ColIdefics3Processor.from_pretrained(MODEL_NAME)

print(" Model & Processor Ready!")

In [ ]:
# --- BƯỚC 3: FUSED INDEXING ---
print(">>> BƯỚC 3: Creating Fused Vectors")

import io
import pickle
import os
import torch
import gc
from PIL import Image
from tqdm.notebook import tqdm

# ==============================================================================
# 1. CẤU HÌNH & CHECKPOINT SETUP
# ==============================================================================
WORKING_DIR = "/kaggle/working"
FINAL_INDEX_PATH = os.path.join(WORKING_DIR, "colsmol_fused_index.pkl")
CHECKPOINT_PATH = os.path.join(WORKING_DIR, "colsmol_checkpoint.pkl") 
BATCH_SIZE = 4 
SAVE_EVERY = 500 

fused_index = []
start_idx = 0
skip_encoding = False 
# nếu đã xong
if os.path.exists(FINAL_INDEX_PATH):
    print(f" Phát hiện file kết quả đã hoàn thành: {FINAL_INDEX_PATH}")
    print(" Đang load index vào RAM...")
    with open(FINAL_INDEX_PATH, 'rb') as f:
        fused_index = pickle.load(f)
    print(f" Đã load {len(fused_index)} layouts. BỎ QUA quá trình Encode.")
    skip_encoding = True

# nếu chưa xong
elif os.path.exists(CHECKPOINT_PATH):
    print(f" Phát hiện Checkpoint dở dang! Đang khôi phục...")
    try:
        with open(CHECKPOINT_PATH, 'rb') as f:
            fused_index = pickle.load(f)
        start_idx = len(fused_index)
        print(f" Đã khôi phục {start_idx} layouts. Tiếp tục từ layout thứ {start_idx}...")
    except Exception as e:
        print(f" Checkpoint lỗi ({e}). Chạy lại từ đầu.")
        fused_index = []
        start_idx = 0
else:
    print(" Chưa có dữ liệu cũ. Chạy mới từ đầu")

# ==============================================================================
# 2. CHUẨN BỊ LOOP 
# ==============================================================================
if not skip_encoding:
    print(f"Tổng số layout cần xử lý: {len(sample_layouts_df)}")
    
    # Cắt DataFrame
    remaining_df = sample_layouts_df.iloc[start_idx:].reset_index(drop=True)
    
    def load_img_safe(row):
        try:
            img = None
            if row['img_type'] == 'path': img = Image.open(row['img_data'])
            elif row['img_type'] == 'binary': img = Image.open(io.BytesIO(row['img_data']))
            
            if img is not None:
                img = img.convert("RGB")
                if img.width < 14 or img.height < 14: return Image.new("RGB", (224, 224), "white")
                return img
        except: pass
        return Image.new("RGB", (224, 224), "white") 

    batch_imgs = []
    batch_texts = [] 

    for i, row in tqdm(remaining_df.iterrows(), total=len(remaining_df), desc="Encoding"):
        
        current_real_idx = start_idx + i
        
        # Prepare Data
        img = load_img_safe(row)
        txt_raw = str(row['final_text'])
        txt = txt_raw.replace("<image>", " ")[:2000] 
        
        batch_imgs.append(img)
        batch_texts.append(txt)
        
        # Process Batch
        if len(batch_imgs) >= BATCH_SIZE or i == len(remaining_df) - 1:
            with torch.no_grad():
                try:
                    inputs_vis = processor.process_images(batch_imgs).to(device)
                    out_vis = model(**inputs_vis)
                    emb_vis_list = list(torch.unbind(out_vis.to("cpu")))
                    
                    inputs_txt = processor.process_queries(batch_texts, max_length=512, suffix="").to(device)
                    out_txt = model(**inputs_txt)
                    emb_txt_list = list(torch.unbind(out_txt.to("cpu")))
                    
                    for k in range(len(emb_vis_list)):
                        v_img = emb_vis_list[k]
                        v_txt = emb_txt_list[k]
                        fused_vec = torch.cat([v_img, v_txt], dim=0)
                        fused_index.append(fused_vec)
                        
                except Exception as e:
                    print(f"\n Lỗi Batch {current_real_idx}: {e}. Skipping...")
            
            batch_imgs = []
            batch_texts = []
            
            if current_real_idx % 100 == 0: 
                torch.cuda.empty_cache()
                gc.collect()

            # Save Checkpoint
            if len(fused_index) % SAVE_EVERY == 0:
                with open(CHECKPOINT_PATH, 'wb') as f: pickle.dump(fused_index, f)

    # LƯU FILE CUỐI CÙNG
    with open(FINAL_INDEX_PATH, 'wb') as f:
        pickle.dump(fused_index, f)
    
    # Xóa checkpoint để tiết kiệm dung lượng
    if os.path.exists(CHECKPOINT_PATH): os.remove(CHECKPOINT_PATH)

print("="*60)
print(f" BƯỚC 3 HOÀN TẤT!")
print(f" Tổng số Docs trong Index: {len(fused_index)}")
print(f" Vị trí file: {FINAL_INDEX_PATH}")
print("="*60)

In [ ]:

# --- BƯỚC 3.5: TRIAL SCORING MODULE ---
# Paper: "TRIAL: Token Relation and Importance for Late-Interaction Retrieval"
# Implements: Token Importance Weights (Sec 3.4) + Bigram Relation Scores (Sec 3.3)
print(">>> BƯỚC 3.5: Initializing TRIAL-Enhanced Scoring...")

import torch
import torch.nn.functional as F
import unicodedata

# ==============================================================================
# STOPWORD SUPPRESSION
# ==============================================================================
_STOPWORDS = {
    # question words
    "what", "how", "when", "where", "which", "who", "whose", "why",
    "according", "describe", "explain",
    # auxiliaries / copulas
    "is", "are", "was", "were", "be", "been", "being",
    "do", "does", "did",
    "have", "has", "had",
    "can", "could", "will", "would", "shall", "should", "may", "might", "must",
    # determiners / articles
    "the", "a", "an",
    # prepositions
    "of", "to", "in", "on", "at", "by", "for", "with", "from", "up",
    "about", "into", "through", "during", "before", "after", "above",
    "below", "between", "out", "off", "over", "under", "again",
    "further", "then", "once",
    # conjunctions
    "and", "or", "but", "nor", "so", "yet", "both", "either",
    "neither", "not", "no", "if", "than", "because", "as", "until",
    "while", "although", "though", "since", "unless",
    # pronouns / misc
    "it", "its", "this", "that", "these", "those", "there", "their",
    "they", "them", "he", "she", "we", "you", "i", "my", "your",
    "his", "her", "our", "any", "all", "each", "every", "more",
    "most", "other", "such", "same", "own", "s",
    # punctuation tokens
    ",", ".", "?", "!", ":", ";", "'", '"', "(", ")", "[", "]",
    "-", "_", "/", "\\", "'s", "n't",
}

# Unicode "mojibake" fragments produced when tokenizing curly quotes / dashes.
# e.g. U+2019 (') → b'\xe2\x80\x99' → tokens 'â', 'Ģ', 'Ļ' in some vocabs.
# Includes both individual byte fragments AND compound forms that may appear
# as a single token depending on the tokenizer vocabulary.
_MOJIBAKE_FRAGMENTS = {
    # Curly apostrophe U+2019 — individual byte fragments
    'â', 'Ģ', 'Ļ',
    # Compound forms (sometimes tokenized as a single unit)
    'âĢĻ', 'âĢ',
    # Em-dash U+2014 fragments
    'ā', 'ĵ', 'āĵ',
    # En-dash U+2013 fragments
    'Ĵ',
    # Left/right double curly quote fragments (U+201C, U+201D)
    'âĢľ', 'âĢĿ',
}


def _clean_token_str(tok_str):
    """Strip BPE space prefix, normalize unicode, return clean string."""
    s = tok_str.replace('\u0120', '').replace('\u2581', '').strip()
    # Normalize curly quotes / smart quotes → ASCII equivalents
    s = s.replace('\u2019', "'").replace('\u2018', "'")
    s = s.replace('\u201c', '"').replace('\u201d', '"')
    s = s.replace('\u2014', '-').replace('\u2013', '-')
    return s


def _token_is_suppressed(tok_str):
    """
    Returns True if this BPE token should be suppressed (weight forced to 0).
    Handles Ġ-prefixed (U+0120) and ▁-prefixed (U+2581) forms.
    Special tokens (<end_of_utterance>, <image>, <|im_end|>, ...) always suppressed.
    Mojibake byte-fragment tokens also suppressed.
    """
    if not tok_str:
        return True
    # Any <...> style special token
    if tok_str.startswith('<') and '>' in tok_str:
        return True
    # Known mojibake fragments (individual or compound)
    if tok_str in _MOJIBAKE_FRAGMENTS:
        return True
    clean = _clean_token_str(tok_str).lower()
    return clean in _STOPWORDS or not clean


def _token_is_numeric_or_operator(clean_str):
    """
    Returns True for purely numeric or operator-heavy tokens (e.g. '=60', '>=', '3.14').

    WHY: These tokens should NOT define the content centroid.
    If a question contains '=60', its unique embedding would single-handedly
    pull the centroid toward itself → cosine(=60, centroid)≈1.0 → captures
    almost all weight after softmax (τ=0.05 is very sharp).

    FIX: Exclude from centroid computation; they still RECEIVE weight via
    cosine similarity to the content centroid built from real content words.
    """
    if not clean_str:
        return False
    # Purely numeric (int, float, with optional comma separators)
    try:
        float(clean_str.replace(',', ''))
        return True
    except ValueError:
        pass
    # Operator-heavy: >70% of chars are digits or math/comparison operators
    op_chars = set('=<>+-*/\\%^&|~@#0123456789.,')
    if len(clean_str) >= 1 and sum(1 for c in clean_str if c in op_chars) / len(clean_str) > 0.7:
        return True
    return False


def _compute_token_weights(q_embs, temperature=0.05, input_ids=None):
    """
    Token importance weights via content-centroid similarity (stopwords suppressed).

    Pipeline:
      1. padding_mask   = tokens with non-zero embedding norm
      2. suppress_mask  = stopwords / punctuation / special tokens / mojibake
      3. active_mask    = padding_mask AND NOT suppress_mask
                          → used for final weight distribution (allows numerics)
      4. centroid_mask  = active_mask AND NOT numeric/operator tokens
                          → used ONLY for centroid computation
                          → prevents '=60' from monopolizing the centroid
      5. centroid       = L2_norm( mean of centroid_mask token embeddings )
      6. sim_to_centroid = cosine(E_qi, centroid)
      7. w_q            = softmax( sim / τ ) over active tokens
      8. weight cap     = clamp(w_q, max=MAX_TOKEN_WEIGHT), re-normalize
                          → prevents any single token from > MAX_TOKEN_WEIGHT
                          → partially rescues secondary keywords like 'colour'

    Returns:
        w_q:         [B, Nq]  importance weights (sums to 1 over active tokens)
        active_mask: [B, Nq]  1 = content token, 0 = suppressed
    """
    B, Nq, D = q_embs.shape
    device = q_embs.device

    # 1. Padding mask
    padding_mask = (torch.norm(q_embs, dim=-1) > 1e-6).float()

    # 2. Suppress mask + numeric mask — string matching on decoded token strings
    suppress_mask = torch.zeros(B, Nq, device=device)
    numeric_mask  = torch.zeros(B, Nq, device=device)   # numeric/operator tokens
    ids_list = None
    if input_ids is not None:
        try:
            ids_list = input_ids.cpu().tolist()
            for b in range(B):
                tok_strs = processor.tokenizer.convert_ids_to_tokens(ids_list[b])
                for j, tok in enumerate(tok_strs):
                    if j >= Nq:
                        break
                    if _token_is_suppressed(tok):
                        suppress_mask[b, j] = 1.0
                    else:
                        clean = _clean_token_str(tok)
                        if _token_is_numeric_or_operator(clean):
                            numeric_mask[b, j] = 1.0
        except Exception as e:
            print(f"  [warn] suppress_mask: {e} — skipping suppression")

    # active_mask: non-suppressed tokens (numerics CAN receive weight)
    active_mask = padding_mask * (1.0 - suppress_mask)

    # Fallback: if every token suppressed, revert to padding mask
    fallback_rows = (active_mask.sum(dim=1) < 1.0)
    if fallback_rows.any():
        active_mask[fallback_rows] = padding_mask[fallback_rows]

    # centroid_mask: active tokens minus numeric/operator (for stable centroid)
    centroid_mask = active_mask * (1.0 - numeric_mask)
    fallback_centroid = (centroid_mask.sum(dim=1) < 1.0)
    if fallback_centroid.any():
        centroid_mask[fallback_centroid] = active_mask[fallback_centroid]

    # 3. Content centroid (from centroid_mask tokens only)
    q_unit = F.normalize(q_embs, dim=-1)
    centroid_exp = centroid_mask.unsqueeze(-1)
    centroid_sum = (q_unit * centroid_exp).sum(dim=1)
    n_centroid = centroid_mask.sum(dim=1, keepdim=True).clamp(min=1.0)
    centroid = F.normalize(centroid_sum / n_centroid, dim=-1).unsqueeze(1)

    # 4. Cosine sim → softmax weights (over ALL active tokens, including numerics)
    sim = (q_unit * centroid).sum(dim=-1)
    sim_masked = sim * active_mask + (1.0 - active_mask) * (-1e9)
    w_q = F.softmax(sim_masked / temperature, dim=-1)

    # 5. Weight cap: no single token can exceed MAX_TOKEN_WEIGHT
    #    This prevents e.g. '=60' from taking 60% weight and partially
    #    rescues secondary-topic keywords like 'colour'.
    w_q = w_q.clamp(max=MAX_TOKEN_WEIGHT)
    # Re-normalize so weights still sum to 1 over active tokens
    w_sum = (w_q * active_mask).sum(dim=-1, keepdim=True).clamp(min=1e-9)
    w_q = w_q * active_mask / w_sum

    return w_q, active_mask


WEIGHT_TEMPERATURE = 0.05
MAX_TOKEN_WEIGHT   = 0.35   # No single token can take more than 35% of total weight


# ==============================================================================
# DISPLAY HELPER — merge BPE subwords into full words
# ==============================================================================
def _merge_subword_weights(tokens, weights):
    """
    Merge BPE/SentencePiece subword pieces into full words for display.

    SentencePiece convention (ColSmolVLM uses Ġ / U+0120):
      Ġword  → space before word → starts a NEW word
      word   → no space prefix   → CONTINUATION subword → merge with previous

    Special tokens (<end_of_utterance>, <|im_end|>, ...) → word boundary, skipped.
    Mojibake fragments (âĢĻ, âĢ, etc.) → suppressed, not appended.
    Zero-weight punctuation continuation subwords → word boundary, not appended.
    """
    merged = []
    cur_word = None
    cur_weight = 0.0

    def _flush():
        nonlocal cur_word, cur_weight
        if cur_word is not None:
            merged.append((cur_word, cur_weight))
            cur_word = None
            cur_weight = 0.0

    for tok, w in zip(tokens, weights):
        if not tok:
            continue

        # Any <...> special token → word boundary, skip
        if tok.startswith('<') and '>' in tok:
            _flush()
            continue

        # Standard padding/EOS tokens
        if tok in ('<pad>', '</s>', '<s>', '<unk>'):
            _flush()
            continue

        # Mojibake fragments (individual or compound) → skip entirely (don't flush)
        if tok in _MOJIBAKE_FRAGMENTS:
            continue

        clean = _clean_token_str(tok)
        if not clean:
            continue

        is_new_word = tok.startswith('\u0120') or tok.startswith('\u2581') or cur_word is None

        if is_new_word:
            _flush()
            cur_word = clean
            cur_weight = w
        else:
            # Continuation subword.
            # Zero-weight punctuation → word boundary (avoid "code," "layer?" artifacts)
            if w < 1e-9 and all(c in '.,?!:;\'"()[]\\/=' for c in clean):
                _flush()
            else:
                cur_word = cur_word + clean
                cur_weight += w

    _flush()
    return merged


# ==============================================================================
# SCORING FUNCTIONS
# ==============================================================================
def trial_score(q_embs, doc_embs_list, lambda_rel=0.5, input_ids=None):
    """
    TRIAL scoring: Score(Q,D) = Σ_i  w_qi * [MaxSim(qi,D) + λ * Rel(qi,D)]

    ▸ Token weights: content-centroid similarity with stopword suppression
    ▸ Relation:      Hadamard bigram product (Sec 3.3)
    """
    B, Nq, D = q_embs.shape
    N = len(doc_embs_list)
    device = q_embs.device

    w_q, _ = _compute_token_weights(q_embs, temperature=WEIGHT_TEMPERATURE,
                                    input_ids=input_ids)

    q_norm_vecs = F.normalize(q_embs, dim=-1)
    has_bigrams = (Nq > 1)
    if has_bigrams:
        q_rel_vecs = q_norm_vecs[:, 1:] * q_norm_vecs[:, :-1]

    scores = torch.zeros(B, N, device=device)

    for d_idx in range(N):
        d_emb = doc_embs_list[d_idx]

        sim_matrix = torch.matmul(q_embs, d_emb.T)
        max_sim, max_j = sim_matrix.max(dim=-1)

        rel_contribution = torch.zeros(B, Nq, device=device)
        if has_bigrams:
            d_norm_vecs = F.normalize(d_emb, dim=-1)
            j_curr = max_j[:, 1:]
            j_prev = max_j[:, :-1]
            d_curr = d_norm_vecs[j_curr.reshape(-1)].reshape(B, Nq - 1, D)
            d_prev = d_norm_vecs[j_prev.reshape(-1)].reshape(B, Nq - 1, D)
            d_rel_vecs = d_curr * d_prev
            rel_contribution[:, 1:] = (q_rel_vecs * d_rel_vecs).sum(dim=-1)

        token_scores = max_sim + lambda_rel * rel_contribution
        scores[:, d_idx] = (w_q * token_scores).sum(dim=-1)

    return scores


def trial_score_with_analysis(q_embs, doc_embs_list, lambda_rel=0.5, input_ids=None):
    """Same as trial_score() but also returns diagnostics dict."""
    B, Nq, D = q_embs.shape
    N = len(doc_embs_list)
    device = q_embs.device

    w_q, _ = _compute_token_weights(q_embs, temperature=WEIGHT_TEMPERATURE,
                                    input_ids=input_ids)

    q_norm_vecs = F.normalize(q_embs, dim=-1)
    has_bigrams = (Nq > 1)
    if has_bigrams:
        q_rel_vecs = q_norm_vecs[:, 1:] * q_norm_vecs[:, :-1]

    scores           = torch.zeros(B, N, device=device)
    maxsim_baseline  = torch.zeros(B, N, device=device)
    rel_scores_total = torch.zeros(B, N, device=device)

    for d_idx in range(N):
        d_emb = doc_embs_list[d_idx]
        sim_matrix = torch.matmul(q_embs, d_emb.T)
        max_sim, max_j = sim_matrix.max(dim=-1)

        rel_contribution = torch.zeros(B, Nq, device=device)
        if has_bigrams:
            d_norm_vecs = F.normalize(d_emb, dim=-1)
            j_curr = max_j[:, 1:]
            j_prev = max_j[:, :-1]
            d_curr = d_norm_vecs[j_curr.reshape(-1)].reshape(B, Nq - 1, D)
            d_prev = d_norm_vecs[j_prev.reshape(-1)].reshape(B, Nq - 1, D)
            d_rel_vecs = d_curr * d_prev
            rel_contribution[:, 1:] = (q_rel_vecs * d_rel_vecs).sum(dim=-1)

        token_scores = max_sim + lambda_rel * rel_contribution
        scores[:, d_idx]           = (w_q * token_scores).sum(dim=-1)
        maxsim_baseline[:, d_idx]  = max_sim.sum(dim=-1)
        rel_scores_total[:, d_idx] = (w_q * rel_contribution).sum(dim=-1)

    diagnostics = {
        'maxsim_baseline': maxsim_baseline,
        'rel_scores':      rel_scores_total,
        'w_q':             w_q,
        'w_q_entropy':     -(w_q * torch.log(w_q + 1e-10)).sum(dim=-1),
        'active_tokens':   (w_q > 1e-4).sum(dim=-1).float(),
    }
    return scores, diagnostics


print("✅ TRIAL Scoring Functions Ready!")
print(f"   • trial_score(q, docs, λ=0.5, input_ids)  — Production mode")
print(f"   • trial_score_with_analysis(...)           — Debug mode")
print(f"   ─────────────────────────────────────────────────────────")
print(f"   Token Importance: content-centroid sim (τ={WEIGHT_TEMPERATURE})")
print(f"     Centroid mask : stopwords + specials + mojibake + numeric/operator")
print(f"     Score mask    : stopwords + specials + mojibake  (numerics CAN score)")
print(f"     Weight cap    : max {MAX_TOKEN_WEIGHT:.0%} per token (re-normalized)")
print(f"   Token Relations : Hadamard bigram (Sec 3.3)")
print(f"   Relation weight : λ = 0.5")


In [ ]:

# --- BƯỚC 4: EVALUATION & REPORTING (TRIAL Fixed Lambda) ---
print(">>> BƯỚC 4: TRIAL Evaluation & Exporting Reports...")

import json
import torch
import torch.nn.functional as F
import pickle
import os
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

# ==============================================================================
# 1. SETUP
# ==============================================================================
WORKING_DIR = "/kaggle/working"
INDEX_PATH = os.path.join(WORKING_DIR, "colsmol_fused_index.pkl")
ANNOTATIONS_PATH = "/kaggle/input/mmdocir-eval-data/MMDocIR_annotations.jsonl"

LAMBDA_REL = 0.5  # Cố định lambda

# Load Index
if 'fused_index' not in globals() or not fused_index:
    if os.path.exists(INDEX_PATH):
        with open(INDEX_PATH, 'rb') as f:
            fused_index = pickle.load(f)
    else:
        raise FileNotFoundError(" Index file not found!")

# Check DataFrame
if 'sample_layouts_df' not in globals():
    raise ValueError("'sample_layouts_df' not found!")

# ==============================================================================
# 2. GENERATE QA PAIRS
# ==============================================================================
print(" Generating QA Pairs...")

def calculate_iou(box1, box2):
    b1 = list(box1) if not isinstance(box1, list) else box1
    b2 = list(box2) if not isinstance(box2, list) else box2

    x1, y1 = max(b1[0], b2[0]), max(b1[1], b2[1])
    x2, y2 = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    union = ((b1[2]-b1[0])*(b1[3]-b1[1])) + ((b2[2]-b2[0])*(b2[3]-b2[1])) - inter
    return inter / union if union > 0 else 0

qa_pairs = []
doc_lookup = {k: v for k, v in sample_layouts_df.groupby('join_doc_name')}
available_docs = set(doc_lookup.keys())
print(f" Docs in RAM: {list(available_docs)}")

matched_docs = 0
with open(ANNOTATIONS_PATH, 'r') as f:
    for line in f:
        try:
            doc_data = json.loads(line)
        except:
            continue

        target_doc = doc_data['doc_name'].replace('.pdf', '')
        if target_doc not in available_docs:
            continue

        matched_docs += 1
        doc_layouts = doc_lookup[target_doc]
        domain = doc_data.get('domain', 'General')
        col_page = 'page_idx' if 'page_idx' in doc_layouts.columns else 'page_id'

        current_doc_df = doc_layouts.copy()
        current_doc_df['safe_page'] = pd.to_numeric(current_doc_df[col_page], errors='coerce').fillna(-999).astype(int)

        for q_item in doc_data['questions']:
            gt_indices = []
            if 'layout_mapping' in q_item:
                for target in q_item['layout_mapping']:
                    t_page, t_bbox = target['page'], target['bbox']
                    try:
                        t_page_int = int(t_page)
                        cands1 = current_doc_df[current_doc_df['safe_page'] == t_page_int]
                        cands2 = current_doc_df[current_doc_df['safe_page'] == t_page_int - 1]
                        cands = pd.concat([cands1, cands2])
                        for idx, row in cands.iterrows():
                            if calculate_iou(row['bbox'], t_bbox) > 0.5:
                                gt_indices.append(int(idx))
                    except:
                        continue

            if gt_indices:
                qa_pairs.append({
                    'question': q_item['Q'],
                    'gt_indices': list(set(gt_indices)),
                    'doc_name': target_doc,
                    'domain': domain
                })

print(f" Đã quét {matched_docs} tài liệu khớp tên.")
print(f" Found {len(qa_pairs)} QA pairs valid for testing.")

# ==============================================================================
# 3. SCORING & EXPORT
# ==============================================================================
if len(qa_pairs) > 0:
    print(f"\n{'='*60}")
    print(f" TRIAL SCORING  λ = {LAMBDA_REL}")
    print(f" Score(Q,D) = Base_MaxSim + {LAMBDA_REL} × Relation")
    print(f"{'='*60}")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    try:
        all_docs_gpu = [d.to(device) for d in fused_index]
    except:
        print(" VRAM Full. Using CPU.")
        all_docs_gpu = fused_index

    correct_1, correct_5, correct_10 = 0, 0, 0
    total = len(qa_pairs)
    report_data = []

    BATCH_Q = 4
    pbar = tqdm(range(0, total, BATCH_Q), desc="Evaluating")

    for i in pbar:
        batch_qa = qa_pairs[i:i+BATCH_Q]
        queries = [q['question'] for q in batch_qa]

        with torch.no_grad():
            q_inputs = processor.process_queries(queries).to(device)
            q_embeddings = model(**q_inputs)
            # Truyền input_ids để suppress special tokens trong weight
            scores = trial_score(
                q_embeddings, all_docs_gpu,
                lambda_rel=LAMBDA_REL,
                input_ids=q_inputs['input_ids']
            )

        for j, score_row in enumerate(scores):
            top_k_indices = torch.topk(score_row, k=10).indices.cpu().tolist()
            gt_set = set(batch_qa[j]['gt_indices'])

            first_hit = -1
            hits_found = []
            for r, idx in enumerate(top_k_indices):
                if idx in gt_set:
                    if first_hit == -1:
                        first_hit = r + 1
                    hits_found.append(idx)

            if first_hit != -1:
                if first_hit <= 1:  correct_1  += 1
                if first_hit <= 5:  correct_5  += 1
                if first_hit <= 10: correct_10 += 1

            recall_score = len(hits_found) / len(gt_set) if len(gt_set) > 0 else 0.0
            report_data.append({
                'lambda_rel': LAMBDA_REL,
                'doc_name': batch_qa[j]['doc_name'],
                'domain': batch_qa[j]['domain'],
                'question': batch_qa[j]['question'],
                'layout_retrieved': str(top_k_indices),
                'gt': str(list(gt_set)),
                'first_hit_rank': first_hit if first_hit != -1 else 'Not Found',
                'recall': round(recall_score, 2),
                'is_perfect': recall_score == 1.0,
            })

        if (i + BATCH_Q) % 20 == 0:
            pbar.set_postfix({'R@10': f"{correct_10 / min(i+BATCH_Q, total) * 100:.1f}%"})

    print("\n" + "="*50)
    print(f" FINAL RESULTS  (λ = {LAMBDA_REL})")
    print("="*50)
    print(f" Recall@1:  {correct_1  / total * 100:.2f}%")
    print(f" Recall@5:  {correct_5  / total * 100:.2f}%")
    print(f" Recall@10: {correct_10 / total * 100:.2f}%")
    print("="*50)

    # EXPORT CSV
    df_report = pd.DataFrame(report_data)
    report_path = os.path.join(WORKING_DIR, f"trial_report_best_lambda_{LAMBDA_REL}.csv")
    df_report.drop(columns=['is_perfect']).to_csv(report_path, index=False, encoding='utf-8-sig')
    print(f"📄 Report saved: {report_path}")

    err_path = os.path.join(WORKING_DIR, f"trial_error_analysis_best_lambda_{LAMBDA_REL}.csv")
    df_report[~df_report['is_perfect']].drop(columns=['is_perfect']).to_csv(err_path, index=False)
    print(f"⚠️  Error analysis saved: {err_path}")

    # ==============================================================================
    # 4. DIAGNOSTIC: Token Importance Weight Visualization
    # ==============================================================================
    N_DIAG = min(5, len(qa_pairs))
    diag_queries = [qa_pairs[i]['question'] for i in range(N_DIAG)]

    print("\n" + "="*60)
    print(f" DIAGNOSTIC: Token Importance Weights ({N_DIAG} ví dụ)")
    print(f"  w(token) = softmax( sim_to_centroid / τ={WEIGHT_TEMPERATURE} )")
    print(f"  Kỳ vọng: thuật ngữ kỹ thuật >> từ chức năng")
    print("="*60)

    with torch.no_grad():
        diag_inputs = processor.process_queries(diag_queries).to(device)
        diag_embs   = model(**diag_inputs).cpu().float()   # [B, Nq, D]

    diag_ids = diag_inputs['input_ids'].cpu()

    d_weights, _ = _compute_token_weights(
        diag_embs,
        temperature=WEIGHT_TEMPERATURE,
        input_ids=diag_ids
    )

    for i, query in enumerate(diag_queries):
        tokens  = processor.tokenizer.convert_ids_to_tokens(diag_ids[i].tolist())
        weights = d_weights[i].tolist()
        n = min(len(tokens), len(weights))

        # Merge BPE subword fragments into full words, then sort by weight
        raw_pairs = [(tokens[j], weights[j]) for j in range(n)]
        merged = _merge_subword_weights(
            [t for t, _ in raw_pairs], [w for _, w in raw_pairs]
        )
        merged.sort(key=lambda x: x[1], reverse=True)

        print(f"\n[Q{i+1}] {query}")
        print(f"  {'─'*56}")
        print(f"  {'Token':<28} {'Weight':>9}   Importance")
        print(f"  {'─'*56}")
        for tok, w in merged[:15]:
            bar = "█" * min(int(w * 800), 30)
            print(f"  {tok:<28} {w:>9.5f}   {bar}")

    print("\n" + "="*60)
    print(" Kỳ vọng: danh từ / thuật ngữ kỹ thuật > giới từ / từ chức năng.")
    print(" Nếu đúng: 'linear_class', 'AP50', 'DETR' phải top-rank.")
    print("="*60)

else:
    print(" Critical Error: Zero QA Pairs found.")
